# #️⃣ Mapas y Tablas Hash - Implementación de Tablas Hash

## Programación III - Lic. en Sistemas

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap10/02_HashTables_Implementacion_Teoria.ipynb)


### 🎯 Objetivos

Al finalizar este notebook el estudiante podrá:
- Explicar el concepto de función hash (código hash + compresión)
- Implementar la función MAD (Multiply-Add-Divide) para compresión
- Resolver colisiones con **encadenamiento separado** (ChainHashMap)
- Resolver colisiones con **direccionamiento abierto** linear probing (ProbeHashMap)
- Analizar el factor de carga y estrategia de rehashing

### 📚 Contenido

📚 *Data Structures and Algorithms in Python* — Goodrich, Tamassia, Goldwasser (Cap. 10 Secc. 2)
  
---

## ⚙️ ¿Cómo funciona una Tabla Hash?

```
Clave (k)  →  [Función Hash]  →  índice j  →  tabla[j]  →  valor
```

Una tabla hash mapea claves a **índices** de un array usando una función hash:
```
h(k) = índice en [0, N-1]
```

### Dos componentes de h(k)

1. **Código hash** `hash_code(k)`: mapea clave a entero (posiblemente grande)
2. **Función de compresión** `compress(i)`: mapea entero a [0, N-1]

```
h(k) = compress(hash_code(k))
```

### La Propiedad Fundamental

> Si `k1 == k2` entonces `h(k1) == h(k2)` (**obligatorio**)  
> Si `h(k1) == h(k2)` no necesariamente `k1 == k2` (**colisión**)

---

## 🔧 Funciones de Compresión

### División Simple
```
compress(i) = i mod N
```
Problema: N debe evitar potencias de 2 o 10.

### MAD (Multiply-Add-Divide) — más robusto
```
compress(i) = ((a·i + b) mod p) mod N
```
donde:
- `p` es número primo > N
- `a` y `b` son enteros aleatorios con `1 ≤ a < p`, `0 ≤ b < p`
- `a ≠ 0`

MAD evita patrones en colisiones y da distribución uniforme.


In [ ]:
# ============================================================
# HashMapBase: Clase base para todas las hash tables (Código 10.4)
# ============================================================
from collections.abc import MutableMapping
import random

class MapBase(MutableMapping):
    class _Item:
        __slots__ = '_key', '_value'
        def __init__(self, k, v):
            self._key = k; self._value = v
        def __eq__(self, other): return self._key == other._key
        def __ne__(self, other): return not (self == other)
        def __lt__(self, other): return self._key < other._key
        def __repr__(self): return f'({self._key!r}: {self._value!r})'

class HashMapBase(MapBase):
    """Clase base abstracta para tablas hash (Código 10.4 del libro)."""
    
    def __init__(self, cap=11, p=109345121):
        """Crea tabla hash vacía.
        cap: capacidad inicial (preferir número primo)
        p:   número primo grande para función MAD
        """
        self._table = cap * [None]     # array de capacidad 'cap'
        self._n = 0                    # número de entradas en el mapa
        self._prime = p                # número primo para MAD
        self._scale = 1 + random.randrange(p - 1)    # a en MAD
        self._shift = random.randrange(p)              # b en MAD
    
    def _hash_function(self, k):
        """Función hash completa: hash_code + compresión MAD."""
        return (hash(k) * self._scale + self._shift) % self._prime % len(self._table)
    
    def __len__(self):
        return self._n
    
    def __getitem__(self, k):
        j = self._hash_function(k)
        return self._bucket_getitem(j, k)       # puede lanzar KeyError
    
    def __setitem__(self, k, v):
        j = self._hash_function(k)
        self._bucket_setitem(j, k, v)
        if self._n > len(self._table) // 2:     # factor de carga > 0.5
            self._resize(2 * len(self._table) - 1)  # aprox. duplicar tamaño
    
    def __delitem__(self, k):
        j = self._hash_function(k)
        self._bucket_delitem(j, k)
        self._n -= 1
    
    def _resize(self, c):
        """Redimensiona tabla a capacidad c (rehashing)."""
        old = list(self.items())        # guardar pares existentes
        self._table = c * [None]        # nueva tabla vacía
        self._n = 0
        for k, v in old:
            self[k] = v                 # reinsertar en nueva tabla
    
    # ---- métodos abstractos (implementados por subclases) ----
    def _bucket_getitem(self, j, k): raise NotImplementedError
    def _bucket_setitem(self, j, k, v): raise NotImplementedError
    def _bucket_delitem(self, j, k): raise NotImplementedError
    def __iter__(self): raise NotImplementedError


# ============================================================
# ChainHashMap: Encadenamiento separado (Código 10.5)
# ============================================================

class ChainHashMap(HashMapBase):
    """Tabla hash con encadenamiento separado (separate chaining).
    
    Cada bucket es un pequeño UnsortedTableMap.
    Complejidad esperada con factor de carga λ:
    - __getitem__:  O(1 + λ) = O(1) si λ es constante
    - __setitem__:  O(1) esperado
    - __delitem__:  O(1) esperado
    """
    
    # Cada bucket es una lista de pares (key, value)
    
    def _bucket_getitem(self, j, k):
        bucket = self._table[j]   # lista de (key, value) o None
        if bucket is None:
            raise KeyError(k)
        for key, value in bucket:
            if key == k:
                return value
        raise KeyError(k)
    
    def _bucket_setitem(self, j, k, v):
        if self._table[j] is None:
            self._table[j] = []
        bucket = self._table[j]
        for i, (key, value) in enumerate(bucket):
            if key == k:
                bucket[i] = (k, v)   # actualizar existente
                return
        bucket.append((k, v))        # nueva entrada
        self._n += 1
    
    def _bucket_delitem(self, j, k):
        bucket = self._table[j]
        if bucket is None:
            raise KeyError(k)
        for i, (key, value) in enumerate(bucket):
            if key == k:
                del bucket[i]
                return
        raise KeyError(k)
    
    def __iter__(self):
        for bucket in self._table:
            if bucket is not None:
                for key, value in bucket:
                    yield key


# ============================================================
# Demo y Visualización de ChainHashMap
# ============================================================
print("=== ChainHashMap: Encadenamiento Separado ===")
chm = ChainHashMap(cap=7)     # tabla pequeña para ver colisiones

claves = [('uno', 1), ('dos', 2), ('tres', 3), ('cuatro', 4),
          ('cinco', 5), ('seis', 6), ('siete', 7)]

for k, v in claves:
    chm[k] = v
    j = chm._hash_function(k)
    print(f"  insert({k!r:8s}, {v}) → bucket {j}")

print(f"\nFactor de carga: {len(chm)}/{len(chm._table)} = {len(chm)/len(chm._table):.2f}")

print("\nBuckets con entradas:")
for i, bucket in enumerate(chm._table):
    if bucket is not None:
        print(f"  Bucket {i}: {bucket}")

print(f"\nBúsqueda chm['tres'] = {chm['tres']}")
del chm['dos']
print(f"Después del del chm['dos']: len = {len(chm)}")


## 🔍 Direccionamiento Abierto: Linear Probing

En el **open addressing**, todas las entradas van directamente en el array principal.
Si hay colisión en el bucket `j`, se **sondea** la siguiente posición.

### Linear Probing
```
h(k, i) = (h(k) + i) mod N    para i = 0, 1, 2, ...
```

### El Problema del "Clustering"
Linear probing puede crear "grupos" de entradas contiguas (clusters primarios),
lo que **degrada** el rendimiento esperado. La solución incluye:
- **Quadratic probing**: `(h(k) + i²) mod N`
- **Double hashing**: `(h(k) + i·h'(k)) mod N`

### Marca de Eliminación (Sentinel)
Al eliminar en open addressing, no podemos dejar `None` porque rompería la
cadena de sondeo. Se usa una **marca especial** `_DEFUNCT`.

### Factor de Carga Recomendado
- **Chaining**: λ < 0.9 (puede ser mayor)
- **Open addressing**: λ < 0.5 (performance degrada rápidamente después de 0.5)


In [ ]:
# ============================================================
# ProbeHashMap: Direccionamiento Abierto con Linear Probing (Código 10.6)
# ============================================================

class ProbeHashMap(HashMapBase):
    """Tabla hash con linear probing (open addressing).
    
    _DEFUNCT: marca que indica una posición que fue eliminada.
    Necesaria para preservar cadenas de sondeo.
    """
    _DEFUNCT = object()   # centinela para posiciones eliminadas
    
    def _is_available(self, j):
        """Retorna True si el bucket j está disponible (vacío o defunct)."""
        return self._table[j] is None or self._table[j] is ProbeHashMap._DEFUNCT
    
    def _find_slot(self, j, k):
        """Busca clave k en tabla comenzando en índice j.
        
        Retorna (success, index) donde:
        - Si k encontrado: (True, índice donde está)
        - Si no encontrado: (False, índice disponible para insertar)
        """
        firstAvail = None
        while True:
            if self._is_available(j):
                if firstAvail is None:
                    firstAvail = j              # guardar primera posición disponible
                if self._table[j] is None:
                    return False, firstAvail    # fin de cluster, no encontrada
            elif k == self._table[j]._key:
                return True, j                  # clave encontrada
            j = (j + 1) % len(self._table)     # linear probing: siguiente posición
    
    def _bucket_getitem(self, j, k):
        found, s = self._find_slot(j, k)
        if not found:
            raise KeyError(k)
        return self._table[s]._value
    
    def _bucket_setitem(self, j, k, v):
        found, s = self._find_slot(j, k)
        if not found:
            self._table[s] = self._Item(k, v)   # nueva entrada
            self._n += 1
        else:
            self._table[s]._value = v            # actualizar existente
    
    def _bucket_delitem(self, j, k):
        found, s = self._find_slot(j, k)
        if not found:
            raise KeyError(k)
        self._table[s] = ProbeHashMap._DEFUNCT   # marcar como defunct
    
    def __iter__(self):
        for j in range(len(self._table)):
            entry = self._table[j]
            if entry is not None and entry is not ProbeHashMap._DEFUNCT:
                yield entry._key


# ============================================================
# Demo ProbeHashMap
# ============================================================
print("=== ProbeHashMap: Linear Probing ===")
phm = ProbeHashMap(cap=11)

datos = [(10, 'diez'), (22, 'veintidós'), (31, 'treinta y uno'),
         (4,  'cuatro'), (15, 'quince'), (28, 'veintiocho'),
         (17, 'diecisiete'), (88, 'ochenta y ocho'), (59, 'cincuenta y nueve')]

print("Insertando entradas:")
for k, v in datos:
    phm[k] = v
    j = phm._hash_function(k)
    # encontrar posición real
    found, s = phm._find_slot(phm._hash_function(k), k)
    print(f"  insert({k:3d}) → hash={j}, pos_real={s} {'(colisión!)' if j != s else ''}")

print(f"\nFactor de carga: {len(phm)}/{len(phm._table)} = {len(phm)/len(phm._table):.2f}")
print(f"phm[28] = {phm[28]}")

# Eliminación y búsqueda posterior
phm[17]    # confirmar existencia
del phm[17]
print(f"\nDespués del del phm[17]:")
print(f"  phm.get(17, 'NO EXISTE') = {phm.get(17, 'NO EXISTE')}")
print(f"  phm[88] = {phm[88]}  (buscó pasando por el defunct)")


## 📊 Comparación: Chaining vs Open Addressing

| Aspecto | Chaining | Linear Probing |
|---------|----------|----------------|
| Estructura | Array de listas | Array plano |
| Factor de carga máx recomendado | ~0.9 | ~0.5 |
| Memoria extra | Punteros de listas | Solo el array |
| Cache performance | Malo (listas dispersas) | Excelente (array contiguo) |
| Eliminación | Simple (eliminar de lista) | Requiere marca `_DEFUNCT` |
| Peor caso | O(n) | O(n) |
| Caso esperado | O(1) | O(1) si λ < 0.5 |

### Python's `dict` y `set`
Python usa **open addressing** con una variante de **double hashing**.
Desde Python 3.6, los diccionarios también son **ordenados por inserción**.

### Rehashing
Cuando λ > umbral:
1. Crear nueva tabla con aprox. **doble** de capacidad (preferir primo)
2. Reinsertar todas las entradas en la nueva tabla
- Costo: O(n) por rehashing, pero amortizado O(1) por operación


In [ ]:
# ============================================================
# Análisis de rendimiento: Chaining vs ProbeHashMap vs dict
# ============================================================
import time, random

def benchmark_hashmap(cls, n):
    """n inserciones + n búsquedas aleatorias."""
    m = cls()
    keys = list(range(n))
    random.shuffle(keys)
    
    t0 = time.perf_counter()
    for k in keys:
        m[k] = k
    t_ins = (time.perf_counter() - t0) * 1000
    
    t0 = time.perf_counter()
    for k in random.sample(keys, min(n, 500)):
        _ = m[k]
    t_get = (time.perf_counter() - t0) * 1000
    
    return t_ins, t_get

print(f"{'N':>5} | {'ChainHashMap ins':>16} | {'ProbeHashMap ins':>16} | {'dict ins':>10} |")
print("-" * 60)
for n in [200, 500, 1000]:
    t_c, _ = benchmark_hashmap(ChainHashMap, n)
    t_p, _ = benchmark_hashmap(ProbeHashMap, n)
    t_d, _ = benchmark_hashmap(dict, n)
    print(f"{n:>5} | {t_c:>14.2f}ms | {t_p:>14.2f}ms | {t_d:>8.2f}ms |")

print("\n💡 dict de Python es ~10-100x más rápido que nuestras implementaciones")
print("   (C optimizado, tabla de tamaño óptimo, hash specializado por tipo)")

print("\n=== Propiedades de hashabilidad en Python ===")
print(f"hash('python')     = {hash('python')}")
print(f"hash(42)           = {hash(42)}")
print(f"hash((1, 2, 3))    = {hash((1, 2, 3))}")
print(f"hash(3.14)         = {hash(3.14)}")

try:
    hash([1, 2, 3])
except TypeError as e:
    print(f"hash([1,2,3])      → TypeError: {e}")

try:
    hash({'a': 1})
except TypeError as e:
    print(f"hash({{'a':1}})      → TypeError: {e}")
    
print("\n💡 Solo objetos inmutables son hasheables (int, str, tuple, frozenset)")


---

🔗 **Referencia:** Goodrich et al., Cap. 10.2

---

![Creative Commons](https://mirrors.creativecommons.org/presskit/buttons/80x15/png/by-sa.png)

© 2026 Cátedra Programación III – Lic. Sistemas – FCAD/UNER

This notebook is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License (CC BY-SA 4.0)

<https://creativecommons.org/licenses/by-sa/4.0/>